# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Here's a clean breakdown of each concept as it applies to the DAS experiment:

---

**Geo Index**

The Geo Index is a composite score (0 to 1) that normalizes where a student's school sits on the rural-to-metro spectrum, adjusted for resource availability. It's built from three inputs: the NCES urbanicity code (rural/town/suburban/urban), the school's Title I status, and the district's `tech_adoption` score. A score near 0 means rural, under-resourced, low-tech. A score near 1 means dense metro, well-funded, high-tech. It's used as a feature in the clustering model so that two students with identical test scores but very different geographic contexts don't end up in the same segment — because their growth trajectories and intervention needs are fundamentally different.

---

**Dropout Rate**

$$\text{Dropout Rate} = \frac{\text{Students who abandoned a session mid-way}}{\text{Total students who started a session}} \times 100$$

In the DAS context, "dropout" means a student exits a practice session before completing it. The control arm recorded 31% — roughly 1 in 3 Seg C students quit before finishing. The treatment arm brought this to 18%.

---

**Score Trajectory**

$$\text{Score Trajectory} = \frac{\text{Score}_{t} - \text{Score}_{t-1}}{\text{Score}_{t-1}} \times 100$$

This is the year-over-year percentage change in a student's average score across subjects. A trajectory of -1.8% means the student is scoring slightly worse each year — not dramatically, but persistently declining. The IRT adaptive path flipped this to +2.4% in the treatment arm.

---

**Stateless Function**

A stateless function produces its output based only on its current inputs — it holds no memory of previous calls. In DAS, the difficulty engine takes in `(student_theta, recent_answers)` and returns `next_difficulty_level`. It doesn't store the student's history inside itself; all state lives in the database. This matters for A/B testing because you can switch the flag on or off per student without side effects leaking between arms — the control group's question delivery is completely untouched.

```
das_engine(student_theta, recent_answers) → next_difficulty_level
```

No session memory. No shared cache. Clean separation between arms.

---

**Power Analysis**

Power analysis answers: "how many students do I need in each arm so I can actually detect the effect if it exists?" It's calculated before the experiment runs. The four inputs are:

$$n = \frac{2(z_{\alpha/2} + z_{\beta})^2 \cdot p(1-p)}{(p_1 - p_2)^2}$$

| Input | Value used |
|---|---|
| α (false positive rate) | 0.05 |
| β (false negative rate) | 0.20 → power = 0.80 |
| Control dropout rate p₁ | 0.31 |
| Expected treatment rate p₂ | 0.18 |

Plugging in gives roughly 900 students per arm (9,084 / 9,110 in the actual run — oversampled for buffer). Running with too few students means you might see a real effect and still fail to reach significance. Power analysis prevents that waste.

---

**Randomisation to Prevent Confounding**

Plain random assignment can accidentally cluster all high-performing teachers into one arm, or put all grade 8 students in treatment. Stratified randomisation prevents this by splitting the sample within each stratum before assigning. The strata used in DAS were school and grade:

```
For each (school × grade) bucket:
  → randomly assign 50% to Control
  → randomly assign 50% to Treatment
```

This guarantees that whatever is special about "Jefferson Middle, Grade 9" — teacher quality, bell schedule, local culture — is equally represented in both arms. Without this, a confound could explain the entire result.

---

**pre\_score, post\_score, delta\_score**

These are the three columns in `ab_assignments` that capture the measurable academic effect of the experiment.

`pre_score` is the student's average score across subjects in the 8 weeks before the experiment started. It's the baseline — locked at assignment time and never updated.

`post_score` is the student's average score across the same subjects in the 8 weeks of the experiment window.

`delta_score` is the arithmetic difference:

$$\text{delta\_score} = \text{post\_score} - \text{pre\_score}$$

A positive delta means the student improved during the experiment period. A negative delta means they declined. The DAS result: control arm averaged delta = -1.8 (continuing their existing decline), treatment arm averaged delta = +2.4 (a reversal of trend). The difference between those two deltas — 4.2 points — is the treatment effect attributed to IRT adaptive difficulty.

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Q: will this be a paired t test or independent t test. if we have pre and post scores for per student.

Great question — this is a nuanced decision worth thinking through carefully.

---

**Short answer: both, for different things.**

---

**Paired t-test — for within-arm improvement**

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}$$

Where $\bar{d}$ is the mean of `delta_score` (post − pre) per student, and $s_d$ is the standard deviation of those deltas.

You use this when asking: *"Did students in the treatment arm actually improve from their own baseline?"* Each student is compared to themselves, so pre-existing differences in ability cancel out. This is the right test for:

- Did treatment students get better? (`delta_score` in treatment arm, tested against 0)
- Did control students get worse? (`delta_score` in control arm, tested against 0)

The pairing is one student's pre vs their own post. You have exactly that in `ab_assignments` — one row per student with both scores.

---

**Independent (two-sample) t-test — for the causal A/B comparison**

$$t = \frac{\bar{X}_{treatment} - \bar{X}_{control}}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

You use this when asking: *"Did the treatment arm improve more than the control arm?"* The two groups are independent — a student is in one arm or the other, never both. This is the core A/B comparison:

- Compare `delta_score` of treatment vs `delta_score` of control
- Compare dropout rate of treatment vs control

The two groups have no natural pairing between them — student 47 in treatment is not matched to any specific student in control.

---

**The right analytical flow for DAS**

| Question | Test | Input |
|---|---|---|
| Did treatment improve from baseline? | Paired t-test | `pre_score` vs `post_score` within treatment arm |
| Did control decline from baseline? | Paired t-test | `pre_score` vs `post_score` within control arm |
| Did treatment improve *more* than control? | Independent t-test | `delta_score` treatment vs `delta_score` control |
| Did dropout differ between arms? | Independent t-test (proportions) | Dropout rate treatment vs control |

---

**Why not just use a paired test for the A/B comparison?**

Because the students in arm A are not matched to specific students in arm B. Pairing requires a logical relationship between each observation pair — same student before/after, or twin studies, or matched cohorts. Randomly assigned groups have no such link. Forcing a pairing where none exists inflates degrees of freedom incorrectly and gives you a misleading p-value.

---

**One refinement worth making**

Since you already have `pre_score` per student, the cleanest approach for the A/B comparison is actually an **ANCOVA** (Analysis of Covariance) — run an independent comparison of `post_score` between arms while controlling for `pre_score` as a covariate. This is statistically more powerful than comparing `delta_score` directly, because it accounts for regression to the mean (very low-scoring students tend to improve regardless of treatment). In practice for this experiment size (~9,000 per arm), the difference is small — but it's the technically correct choice.

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Here's the full interactive dashboard — sample dataset of 20 students, live computed tests, and the Python code side by side.The dashboard computes all three tests live from the actual numbers in the table. A few things worth noting about the dataset design:

The 10 control students all have small negative deltas (post < pre), reflecting the real-world Seg C trend of gradual decline. The 10 treatment students all have positive deltas, reflecting the IRT adaptive effect. This makes the independent t-test pick up a clean, significant separation between the two groups.

The paired t-test on the control arm should come back significant (p < 0.05) with a negative t-statistic — confirming the decline is real, not noise. The paired t-test on the treatment arm should also come back significant, but with a positive t-statistic — confirming genuine improvement. The independent t-test then answers the causal question: is the treatment delta meaningfully larger than the control delta?

On the Python code — two things to note. `stats.ttest_rel` is the paired test; it takes two arrays of equal length where index position defines the pair (student's pre vs their own post). `stats.ttest_ind` is the independent test; `equal_var=False` switches it to Welch's variant, which doesn't assume the two groups have the same variance — the safer default when you can't verify that assumption. Cohen's d is computed manually since scipy doesn't return it directly; a value above 0.8 is conventionally considered a large effect.

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

Q: why should the delta_score for control group be negative?

Great catch — and this is an important distinction worth being precise about.

---

**The short answer: it doesn't have to be negative. It depends on what question you're asking.**

---

**What the dataset was modelling**

The sample dataset was built to mirror the real-world Seg C (At-Risk Rebuilders) finding from the DAS experiment, where the control arm showed a score trajectory of -1.8% per year. So the control students were given small negative deltas on purpose — to simulate a group that was already in gradual decline before the experiment, and continued declining without the intervention.

This is a design choice for the simulation, not a statistical requirement.

---

**In a real experiment, the control delta could go three ways**

| Control delta | What it means |
|---|---|
| Negative (e.g. -2.1) | Students declined even with standard curriculum. This is what Seg C showed historically. |
| Near zero (e.g. +0.3) | Students stayed roughly flat — no meaningful change either way. Most common in stable populations. |
| Positive (e.g. +3.0) | Students improved on their own — due to natural learning, maturation, or a strong baseline curriculum. |

All three are valid. The control group is not supposed to decline — it's supposed to represent what happens with no intervention. Whatever that natural trajectory is, it becomes your baseline to beat.

---

**Why this matters for interpreting the independent t-test**

The independent t-test compares `mean(delta_treatment)` vs `mean(delta_control)`. What you actually care about is the gap between the two:

If control delta = -2.0 and treatment delta = +2.4, the gap is **4.4 points** — partly because treatment helped, partly because control declined.

If control delta = +1.5 and treatment delta = +5.0, the gap is **3.5 points** — treatment still wins, but the story is different. Both groups improved; the adaptive path just accelerated it.

Conflating a negative control delta with "the experiment worked" is a mistake. The treatment effect is always the difference between the two arms, not the absolute value of either one in isolation.

---

**The paired t-test on the control arm is still valid regardless**

Even if the control delta is near zero, running `ttest_rel(pre, post)` on the control arm is useful — it tells you whether the control group's change (however small) is statistically distinguishable from zero, or just random fluctuation. A non-significant result there actually strengthens your experiment: it means control students didn't naturally improve on their own, so any improvement you see in treatment is more credibly attributed to DAS.

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%